# Construindo uma Mini LLM com Poucos Textos

**Aplicações de Negócio de IA — FGV**

Este notebook faz parte de uma série de 13 notebooks sobre aplicações de negócio de Inteligência Artificial. Aqui, o objetivo é **desmistificar o funcionamento interno de um LLM (Large Language Model)** — o tipo de modelo por trás de ferramentas como GPT, Claude, Gemini etc. — construindo, do zero, uma versão minúscula, treinada com um corpus de texto pequeno que caberá em algumas linhas de código.

**Aviso importante:** este é um exercício de **brinquedo (toy example)**. O modelo que vamos treinar tem alguns milhares de parâmetros e aprende com poucos parágrafos de texto. Um LLM comercial real tem **bilhões de parâmetros** e é treinado com **trilhões de tokens** de texto (praticamente toda a internet, livros, código etc.), usando técnicas de tokenização e arquiteturas (Transformers) muito mais sofisticadas do que as que usaremos aqui. Ainda assim, o **princípio central é o mesmo**, e é isso que queremos entender.

**Limitações deste notebook**

Antes de seguir, vale deixar claro o que este notebook **não** é — para evitar comparações injustas com os LLMs comerciais que ele usa como referência:

- **Corpus minúsculo:** o texto de treino é composto por poucos parágrafos, escritos diretamente no código (menos de 2 mil caracteres). LLMs reais (GPT, Claude, Gemini etc.) são treinados com **trilhões de tokens** — uma diferença de muitas ordens de grandeza, não apenas de "um pouco mais de dados".
- **Modelo em nível de caractere, não Transformer:** aqui prevemos um caractere por vez, com uma rede pequena (Embedding + LSTM). Isso é bem diferente da arquitetura **Transformer com atenção** usada pelos LLMs modernos. A semelhança entre os dois é **conceitual** — ambos preveem o próximo elemento de uma sequência — e não arquitetural.
- **Texto gerado imperfeito é esperado, não é um bug:** o modelo vai produzir frases quebradas e, às vezes, sem sentido. Isso é **intencional e didático**: mostra concretamente como um modelo tão pequeno, treinado com tão poucos dados, generaliza mal — não é algo a "corrigir" ajustando hiperparâmetros.
- **Objetivo é a mecânica, não a qualidade do texto:** a ideia aqui é entender, em miniatura, os passos de tokenização, janelas de contexto e amostragem por temperatura que estão por trás de qualquer modelo de linguagem — não produzir texto útil ou de qualidade.

## 1. O que é um "modelo de linguagem"?

No fundo, um modelo de linguagem é apenas uma **função matemática que aprende a estimar uma probabilidade**:

$$P(\text{próximo token} \mid \text{texto anterior})$$

Ou seja: dado um pedaço de texto ("contexto"), o modelo aprende a prever qual é o próximo "pedaço" (token) mais provável de vir a seguir. Um "token" pode ser:

- um **caractere** (o que faremos aqui, por simplicidade didática);
- uma **palavra inteira**;
- ou, como fazem os LLMs comerciais, um **sub-palavra** obtido por um algoritmo de tokenização como o **BPE (Byte Pair Encoding)** — um meio-termo eficiente entre caractere e palavra.

Quando você conversa com o ChatGPT, o Claude ou o Gemini, por trás dos panos o que está acontecendo é, repetidamente:

1. o modelo recebe todo o texto da conversa até aquele ponto;
2. ele calcula a distribuição de probabilidade sobre o **próximo token possível** no vocabulário (que pode ter dezenas de milhares de tokens);
3. ele **amostra** (sorteia, de forma mais ou menos aleatória) um token dessa distribuição;
4. o token escolhido é anexado ao texto, e o processo se repete — um token de cada vez — até formar a resposta inteira.

A diferença entre o que faremos aqui e um LLM real é **essencialmente de escala e sofisticação arquitetural**, não de princípio:

| | Mini LLM (este notebook) | LLM comercial real (GPT, Claude, ...) |
|---|---|---|
| Tokenização | caractere a caractere | sub-palavras (BPE ou similar), vocabulário de ~50 mil a 200 mil tokens |
| Parâmetros | poucos milhares | dezenas de bilhões a mais de um trilhão |
| Dados de treino | alguns parágrafos (~1 KB) | trilhões de tokens (~petabytes de texto) |
| Arquitetura | Embedding + LSTM (rede recorrente) | Transformer com atenção (*self-attention*), dezenas de camadas |
| Hardware de treino | um notebook, poucos segundos de CPU | milhares de GPUs/TPUs, semanas a meses |
| Qualidade do texto gerado | frases quebradas, pouco coerentes | texto fluente, coerente, capaz de raciocínio |

Vamos construir o nosso, passo a passo.

**0. Instalação de dependências**

Este notebook usa `numpy`, `pandas`, `scikit-learn`, `matplotlib` e `tensorflow`. Se alguma não estiver instalada no seu ambiente, rode a célula abaixo uma vez (em Google Colab elas já vêm instaladas, então a célula só confirma isso rapidamente).

In [1]:
%pip install -q numpy pandas scikit-learn matplotlib tensorflow
print('Dependencias OK.')


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: /private/tmp/claude-501/-Users-marcofidos-GitHub/eeac8253-df86-407b-b3e0-06b2333596a4/scratchpad/nbviz-venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Dependencias OK.


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# reprodutibilidade
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow versão:", tf.__version__)

TensorFlow versão: 2.21.0


## 2. O corpus de treino

Diferente de um LLM real (treinado com bilhões de documentos da internet), o nosso corpus é um texto curtíssimo, **escrito diretamente no código**, sobre o próprio tema deste curso: IA aplicada a negócios.

Repare que o texto é pequeno o suficiente para caber em algumas linhas — isso é proposital: queremos que o treinamento rode em segundos em uma CPU comum, e que fique óbvio o quão pouco "conhecimento" um modelo desse tamanho consegue absorver.

In [3]:
corpus = """
A inteligencia artificial esta mudando a forma como as empresas tomam decisoes.
Um modelo de linguagem aprende a prever a proxima palavra ou letra de um texto,
usando padroes que encontrou durante o treinamento com muitos exemplos.

Bancos usam modelos de aprendizado de maquina para detectar fraudes em tempo real.
Lojas usam sistemas de recomendacao para sugerir produtos que o cliente pode gostar.
Hospitais usam visao computacional para ajudar medicos a analisar exames de imagem.

Um chatbot de atendimento ao cliente tambem e, no fundo, um modelo de linguagem.
Ele recebe a mensagem da pessoa como contexto e gera uma resposta, token por token,
tentando prever qual e a sequencia de palavras mais provavel e mais util para ajudar.

Quanto mais dados de qualidade um modelo recebe durante o treinamento, e quanto
maior for a sua capacidade, melhor ele consegue captar os padroes da linguagem
e gerar textos coerentes, uteis e adequados ao contexto de cada aplicacao.

Neste notebook treinamos uma versao muito pequena desse tipo de modelo, apenas
para entender o principio basico por tras de ferramentas muito maiores e mais
poderosas usadas hoje em aplicacoes reais de negocio.
""".strip()

print(f"Tamanho do corpus: {len(corpus)} caracteres")
print()
print(corpus[:300], "...")

Tamanho do corpus: 1183 caracteres

A inteligencia artificial esta mudando a forma como as empresas tomam decisoes.
Um modelo de linguagem aprende a prever a proxima palavra ou letra de um texto,
usando padroes que encontrou durante o treinamento com muitos exemplos.

Bancos usam modelos de aprendizado de maquina para detectar fraudes ...


## 3. Tokenização em nível de caractere

Aqui está a primeira grande simplificação em relação a um LLM real: em vez de usar um tokenizador de sub-palavras (BPE), vamos tratar **cada caractere** como um "token". Isso torna o vocabulário minúsculo (algumas dezenas de símbolos) e o código bem mais simples de entender, ao custo de o modelo ter que aprender padrões de sequências muito mais longas para "entender" uma palavra inteira.

Vamos montar:
- o **vocabulário**: conjunto de caracteres únicos do corpus;
- o mapeamento **char → índice** (`char2idx`), usado para transformar texto em números;
- o mapeamento **índice → char** (`idx2char`), usado para transformar números de volta em texto.

In [4]:
vocab = sorted(set(corpus))
vocab_size = len(vocab)

char2idx = {ch: i for i, ch in enumerate(vocab)}
idx2char = {i: ch for i, ch in enumerate(vocab)}

print(f"Tamanho do vocabulario: {vocab_size} caracteres unicos")
print("Vocabulario:", vocab)

Tamanho do vocabulario: 36 caracteres unicos
Vocabulario: ['\n', ' ', ',', '.', 'A', 'B', 'E', 'H', 'L', 'N', 'Q', 'U', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'x', 'z']


## 4. Preparando os dados de treino: janelas deslizantes

Um modelo de linguagem é treinado com pares **(contexto, próximo token)**. Vamos gerar esses pares deslizando uma janela de tamanho fixo (`SEQ_LEN` caracteres) sobre o corpus inteiro:

- entrada: os `SEQ_LEN` caracteres de uma posição `i` até `i + SEQ_LEN`;
- alvo: o caractere na posição `i + SEQ_LEN` (o "próximo caractere" logo depois da janela).

Deslizamos essa janela um caractere de cada vez por todo o corpus, gerando milhares de exemplos de treino a partir de um texto de poucos parágrafos.

In [5]:
SEQ_LEN = 30  # tamanho da janela de contexto, em caracteres

corpus_idx = [char2idx[ch] for ch in corpus]

X_list, y_list = [], []
for i in range(len(corpus_idx) - SEQ_LEN):
    X_list.append(corpus_idx[i:i + SEQ_LEN])
    y_list.append(corpus_idx[i + SEQ_LEN])

X = np.array(X_list)
y = np.array(y_list)

print(f"Numero de exemplos de treino: {X.shape[0]}")
print(f"Formato de X (exemplos, SEQ_LEN): {X.shape}")
print(f"Formato de y (exemplos,): {y.shape}")

# exemplo de um par (contexto -> proximo caractere)
exemplo_texto = "".join(idx2char[i] for i in X[0])
print()
print(f"Contexto: {exemplo_texto!r}")
print(f"Proximo caractere esperado: {idx2char[y[0]]!r}")

Numero de exemplos de treino: 1153
Formato de X (exemplos, SEQ_LEN): (1153, 30)
Formato de y (exemplos,): (1153,)

Contexto: 'A inteligencia artificial esta'
Proximo caractere esperado: ' '


## 5. O modelo: Embedding + LSTM + Dense (softmax)

Nossa mini rede neural tem três camadas:

1. **Embedding**: transforma cada índice de caractere em um vetor denso de poucas dimensões (aprendido durante o treino) — uma versão minúscula do que os LLMs reais fazem ao representar tokens em espaços de centenas ou milhares de dimensões.
2. **LSTM** (*Long Short-Term Memory*): uma rede neural recorrente que "lê" a sequência de embeddings caractere a caractere, mantendo uma memória interna do que já viu. É a "engenharia" que permite ao modelo levar em conta o contexto anterior. (LLMs modernos usam **Transformers com atenção** em vez de LSTM, mas o objetivo — processar sequência e capturar dependências — é o mesmo.)
3. **Dense + softmax**: transforma a saída da LSTM em uma distribuição de probabilidade sobre todo o vocabulário — ou seja, "qual a chance de cada caractere possível ser o próximo?".

Note como o modelo é propositalmente pequeno: poucas dimensões de embedding, poucas unidades de LSTM. Isso mantém o treino rápido (segundos em CPU) às custas de uma capacidade de aprendizado muito limitada — exatamente o ponto que queremos ilustrar.

In [6]:
EMBED_DIM = 16
LSTM_UNITS = 64

model = keras.Sequential([
    layers.Input(shape=(SEQ_LEN,)),
    layers.Embedding(input_dim=vocab_size, output_dim=EMBED_DIM),
    layers.LSTM(LSTM_UNITS),
    layers.Dense(vocab_size, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 30, 16)         │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        20,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 36)             │         2,340 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,652 (92.39 KB)

 Trainable params: 23,652 (92.39 KB)

 Non-trainable params: 0 (0.00 B)

## 6. Treinando o modelo

Vamos treinar por um número pequeno de épocas — o suficiente para o modelo sair de "chutando aleatoriamente" para "captando algum padrão estatístico do português usado no corpus" (por exemplo, que depois de certas letras costuma vir espaço, ou que certas sequências de letras formam palavras que aparecem no texto).

Não esperamos, de jeito nenhum, que o modelo "aprenda português" de verdade — ele só viu ~1 KB de texto.

In [7]:
history = model.fit(
    X, y,
    epochs=60,
    batch_size=64,
    verbose=2,
)

Epoch 1/60


19/19 - 1s - 30ms/step - accuracy: 0.1240 - loss: 3.5282


Epoch 2/60


19/19 - 0s - 5ms/step - accuracy: 0.1466 - loss: 3.0494


Epoch 3/60


...

19/19 - 0s - 4ms/step - accuracy: 0.3868 - loss: 1.9885


Epoch 59/60


19/19 - 0s - 4ms/step - accuracy: 0.3877 - loss: 1.9758


Epoch 60/60


19/19 - 0s - 4ms/step - accuracy: 0.3938 - loss: 1.9623


## 7. Gerando texto: amostragem e "temperatura"

Para gerar texto novo, repetimos o processo básico de um LLM:

1. damos um texto inicial (**prompt**);
2. o modelo calcula a distribuição de probabilidade do próximo caractere;
3. **amostramos** um caractere dessa distribuição (não necessariamente o mais provável!);
4. anexamos o caractere escolhido ao texto e repetimos, usando os últimos `SEQ_LEN` caracteres como novo contexto.

O parâmetro de **temperatura** controla o quão "ousada" é essa amostragem, reescalando a distribuição de probabilidades antes de sortear:

- **Temperatura baixa** (ex: 0.5): a distribuição fica mais "concentrada" nos caracteres mais prováveis → texto mais previsível, mais repetitivo, mais conservador.
- **Temperatura alta** (ex: 1.0 ou mais): a distribuição fica mais "achatada" (mais uniforme) → texto mais aleatório, mais "criativo", mas também com mais risco de virar ruído sem sentido.

Esse mesmo mecanismo de temperatura existe nos LLMs comerciais — muitas APIs de chat (incluindo a da Anthropic e a da OpenAI) expõem um parâmetro `temperature` exatamente com esse efeito.

In [8]:
def sample_com_temperatura(probs, temperature=1.0):
    """Recebe um vetor de probabilidades (softmax) e sorteia um indice,
    reescalando a distribuicao pela temperatura antes de amostrar."""
    probs = np.asarray(probs).astype("float64")
    # evita log(0)
    probs = np.clip(probs, 1e-10, 1.0)
    log_probs = np.log(probs) / temperature
    exp_probs = np.exp(log_probs)
    probs_ajustadas = exp_probs / np.sum(exp_probs)
    return np.random.choice(len(probs_ajustadas), p=probs_ajustadas)


def gerar_texto(modelo, prompt, n_caracteres=150, temperature=1.0):
    """Gera novos caracteres a partir de um prompt, um de cada vez,
    usando o modelo treinado e amostragem com temperatura."""
    # mantem apenas caracteres conhecidos pelo vocabulario
    texto_gerado = "".join(ch for ch in prompt if ch in char2idx)
    if len(texto_gerado) == 0:
        raise ValueError("O prompt nao contem nenhum caractere do vocabulario de treino.")

    for _ in range(n_caracteres):
        # usa (no maximo) os ultimos SEQ_LEN caracteres como contexto
        contexto = texto_gerado[-SEQ_LEN:]
        # se o contexto for menor que SEQ_LEN, preenche a esquerda repetindo o inicio
        if len(contexto) < SEQ_LEN:
            contexto = contexto.rjust(SEQ_LEN, contexto[0])

        entrada = np.array([[char2idx[ch] for ch in contexto]])
        probs = modelo.predict(entrada, verbose=0)[0]

        proximo_idx = sample_com_temperatura(probs, temperature=temperature)
        texto_gerado += idx2char[proximo_idx]

    return texto_gerado

## 8. Gerando texto com temperaturas diferentes

Vamos usar o mesmo prompt curto e gerar continuações com duas temperaturas diferentes, para comparar o efeito na prática.

In [9]:
prompt = "A inteligencia artificial"

texto_temp_baixa = gerar_texto(model, prompt, n_caracteres=200, temperature=0.5)
print("=== Temperatura 0.5 (mais conservadora) ===")
print(texto_temp_baixa)

=== Temperatura 0.5 (mais conservadora) ===
A inteligencia artificial sada ara ase calide a der el maassa a malas e prares er caquar a arato are paras dar e tecorais da arado de alas era tere cares e demo memos e manmais amtar ar em ara ara parar ua assante dara dentod


In [10]:
texto_temp_alta = gerar_texto(model, prompt, n_caracteres=200, temperature=1.0)
print("=== Temperatura 1.0 (mais aleatoria) ===")
print(texto_temp_alta)

=== Temperatura 1.0 (mais aleatoria) ===
A inteligencia artificiali asa ap
anticoa totoda, dues remdendam utocas uutelo cotom ivamonse diqureisdequco lalide ara loco amentuiis ia toqua core usenndeo caatocam comem emas car muedecos uta a apaguasa potos cosuder usaom


## 9. O que observamos?

Olhando para os textos gerados acima, é esperado notar:

- O modelo **não** produz português perfeito nem frases totalmente coerentes — e isso é absolutamente normal, dado que ele tem apenas algumas dezenas de milhares de parâmetros e viu menos de 2 mil caracteres de treino.
- Com **temperatura mais baixa**, o texto tende a repetir trechos e palavras que apareceram bastante no corpus (como "modelo", "de", "para"), porque o modelo fica mais "preso" às opções de maior probabilidade.
- Com **temperatura mais alta**, aparecem mais combinações estranhas de letras e palavras que não existem em português, porque o modelo passa a considerar opções de baixa probabilidade com mais frequência.
- Ainda assim, é possível notar que o modelo aprendeu **algo**: por exemplo, tende a alternar entre vogais e consoantes de forma mais parecida com português do que uma sequência puramente aleatória de caracteres do vocabulário, e às vezes reproduz pedaços de palavras reais do corpus.

Isso ilustra bem por que os LLMs reais precisam de tantos parâmetros e tantos dados: **capturar a estrutura estatística de um idioma inteiro, com todas as suas variações, exige uma capacidade de modelagem ordens de grandeza maior** do que a que construímos aqui.

## 10. Conclusão: o princípio por trás dos LLMs — e sua ligação com negócios

O que fizemos aqui, em miniatura, é **exatamente** o princípio de funcionamento de qualquer LLM moderno:

> Um modelo de linguagem é uma rede neural treinada para, repetidamente, prever o próximo token dado o texto anterior — e gerar texto novo é simplesmente aplicar esse processo de previsão + amostragem em loop, token por token.

As diferenças entre o nosso mini modelo e um LLM comercial (GPT, Claude, Gemini, etc.) são de **escala e engenharia**, não de princípio:

- tokenização por sub-palavras (BPE) em vez de caractere a caractere;
- arquitetura **Transformer** com mecanismos de **atenção**, em vez de LSTM;
- bilhões de parâmetros treinados com trilhões de tokens de texto;
- etapas adicionais de ajuste fino (*fine-tuning*) e alinhamento (*RLHF* / *instruction tuning*) que ensinam o modelo a seguir instruções e ser útil e seguro, além de apenas "prever o próximo token".

Entender esse princípio básico é útil na prática de negócio porque ele explica **comportamentos** que aparecem em várias das aplicações discutidas nesta série de notebooks:

- **Chatbots de atendimento** (notebook 4): geram cada resposta token por token, prevendo a continuação mais provável e útil dada a conversa até aquele ponto — por isso podem "alucinar" informações plausíveis, mas incorretas, quando o contexto ou o treinamento não cobrem bem aquele caso.
- **Geração de conteúdo** (textos de marketing, resumos, e-mails): a "criatividade" observada é, no fundo, o resultado de amostragem com temperatura sobre uma distribuição de probabilidade aprendida — por isso ajustar a temperatura muda o quão conservador ou "criativo" o texto gerado será.
- **Qualidade e confiabilidade dos modelos**: quanto maior e mais bem treinado o modelo (mais parâmetros, mais dados, melhor curadoria), melhor ele capta os padrões da linguagem e do domínio de negócio — o que explica por que empresas investem tanto em dados de qualidade e em modelos maiores ou mais especializados para aplicações críticas.

Na prática, empresas raramente treinam um LLM do zero (isso custa dezenas de milhões de dólares) — em vez disso, usam modelos já treinados por grandes provedores (como os modelos da Anthropic, OpenAI, Google) e os adaptam ao seu contexto via *prompt engineering*, *fine-tuning* ou técnicas de recuperação de informação (RAG), temas que aparecem nos demais notebooks desta série.

**Anexo: Outras Aplicações de Negócio com a Mesma Técnica**

A técnica usada aqui (modelo de linguagem prevendo o próximo token a partir do contexto anterior) é o princípio por trás de diversas aplicações de negócio, em escala muito maior do que este exemplo didático:

### 1. Atendimento ao Cliente
- **Geração automática de respostas de primeiro nível:** sugerir ou gerar respostas para perguntas frequentes de clientes.

### 2. Gestão do Conhecimento
- **Resumo automático de documentos internos:** condensar relatórios, atas e contratos longos em resumos executivos.

### 3. Marketing e E-commerce
- **Geração de descrições de produto em escala:** criar textos de catálogo consistentes para milhares de itens.

### 4. Business Intelligence
- **Geração automática de relatórios narrativos:** transformar tabelas e dashboards de dados em texto explicativo para tomadores de decisão não técnicos.

**Exercício**

1. **Aumente o corpus e observe o efeito na qualidade do texto gerado.** Adicione mais parágrafos à variável `corpus` (pode ser texto seu, em português, sobre qualquer tema) — por exemplo, dobrando ou triplicando o número de caracteres — e re-treine o modelo do zero. O texto gerado ficou perceptivelmente mais coerente, ou a diferença é sutil? Se a melhora foi pequena mesmo aumentando bastante os dados, o que isso sugere sobre a relação entre quantidade de dados e capacidade do modelo (número de parâmetros de `EMBED_DIM` e `LSTM_UNITS`)?

2. **Varie a temperatura para extremos e compare coerência vs. diversidade.** Gere texto com `temperature=0.1` (bem baixa) e com `temperature=2.0` (bem alta), usando o mesmo prompt. Compare os dois textos gerados: qual repete mais palavras/trechos do corpus original? Qual produz combinações de letras que claramente não existem em português? Em que ponto (entre 0.1 e 2.0) você diria que o texto gerado deixa de parecer português "quebrado" e passa a parecer ruído quase aleatório?</cell id="cell-23">
